# 01 — Dataset Quality Audit

**Building Extraction Module — Master Thesis**

This notebook performs a concise quality check of the datasets used by the Buildings module.

### Main outputs
- Compact dataset-quality summary
- Background-tile distribution
- Representative image–mask quality check
- Final dataset status: **PASS** or **REVIEW REQUIRED**

The notebook does not modify the original datasets.

In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")

# ============================================================
# PROJECT CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(
    r"D:\Ali Chaaban Thesis Project\GeoAI_Thesis_Codebase"
)

BUILDINGS_ROOT = PROJECT_ROOT / "Buildings"

EXPERIMENTS_ROOT = (
    BUILDINGS_ROOT
    / "outputs"
    / "experiments"
)

OUTPUT_DIR = (
    BUILDINGS_ROOT
    / "outputs"
    / "notebooks"
    / "01_dataset_quality"
)

FIGURES_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RANDOM_SEED = 42
SAMPLE_COUNT = 4

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Buildings module:")
print(BUILDINGS_ROOT)

print("\nExperiments folder:")
print(EXPERIMENTS_ROOT)

print("\nNotebook output:")
print(OUTPUT_DIR)

## 1. Discover Training and Validation Data

The notebook automatically reads the `train_manifest.csv` and `val_manifest.csv` files already generated by the module.

This ensures that the quality analysis uses the exact same samples used during model training.

In [ ]:
def discover_manifests(root: Path):

    rows = []

    if not root.exists():
        raise FileNotFoundError(
            f"Experiments folder not found:\n{root}"
        )

    for manifest in sorted(
        root.rglob("*_manifest.csv")
    ):

        name = manifest.name.lower()

        if name == "train_manifest.csv":
            split = "Train"

        elif name == "val_manifest.csv":
            split = "Validation"

        else:
            continue

        rows.append(
            {
                "experiment_id":
                    manifest.parent.name,

                "split":
                    split,

                "manifest_path":
                    str(manifest),
            }
        )

    return pd.DataFrame(rows)


manifest_index = discover_manifests(
    EXPERIMENTS_ROOT
)

if manifest_index.empty:

    raise FileNotFoundError(
        "No train_manifest.csv or "
        "val_manifest.csv files were found."
    )


display(manifest_index)

## 2. Robust Raster Reader

Some ArcGIS-exported TIFF files cannot always be read directly by Pillow.

The following reader attempts:

1. Pillow  
2. Rasterio  
3. ArcPy  

This improves compatibility with the datasets used by the module.

In [ ]:
def read_raster(
    path: Path,
    is_mask=False
):

    path = Path(path)

    # ========================================================
    # 1. Pillow
    # ========================================================

    try:

        with Image.open(path) as img:

            if is_mask:
                return np.asarray(img)

            return np.asarray(
                img.convert("RGB")
            )

    except Exception:
        pass


    # ========================================================
    # 2. Rasterio
    # ========================================================

    try:

        import rasterio

        with rasterio.open(
            str(path)
        ) as src:

            if is_mask:

                return np.asarray(
                    src.read(1)
                )

            band_count = min(
                3,
                src.count
            )

            arr = src.read(
                list(
                    range(
                        1,
                        band_count + 1
                    )
                )
            )

            if arr.ndim == 3:

                arr = np.transpose(
                    arr,
                    (1, 2, 0)
                )

            if arr.ndim == 2:

                arr = np.repeat(
                    arr[..., None],
                    3,
                    axis=2
                )

            elif arr.shape[2] == 1:

                arr = np.repeat(
                    arr,
                    3,
                    axis=2
                )

            return np.asarray(arr)

    except Exception:
        pass


    # ========================================================
    # 3. ArcPy
    # ========================================================

    import arcpy

    arr = arcpy.RasterToNumPyArray(
        str(path),
        nodata_to_value=0
    )

    arr = np.asarray(arr)

    # -------------------------
    # Mask
    # -------------------------

    if is_mask:

        if arr.ndim == 3:

            if arr.shape[0] <= 10:

                arr = arr[0]

            else:

                arr = arr[..., 0]

        return arr


    # -------------------------
    # RGB image
    # -------------------------

    if arr.ndim == 2:

        arr = np.repeat(
            arr[..., None],
            3,
            axis=2
        )

    elif arr.ndim == 3:

        # ArcPy usually:
        # Bands x Height x Width

        if arr.shape[0] <= 10:

            arr = np.transpose(
                arr[:3],
                (1, 2, 0)
            )

        else:

            arr = arr[..., :3]


    if (
        arr.ndim == 3
        and arr.shape[2] == 1
    ):

        arr = np.repeat(
            arr,
            3,
            axis=2
        )

    return arr

## 3. Dataset Quality Audit

The audit checks:

- image availability;
- label availability;
- readability;
- image–label spatial agreement;
- foreground building coverage;
- percentage of background-only tiles.

In [ ]:
IMAGE_COLUMNS = [
    "image_path",
    "image",
    "img_path",
    "img"
]

LABEL_COLUMNS = [
    "label_path",
    "label",
    "mask_path",
    "mask"
]


def find_column(
    columns,
    candidates
):

    lookup = {
        str(column).lower():
        column

        for column in columns
    }

    # Exact match
    for candidate in candidates:

        if candidate in lookup:
            return lookup[candidate]


    # Flexible match
    for column in columns:

        column_lower = str(
            column
        ).lower()

        for candidate in candidates:

            token = candidate.replace(
                "_path",
                ""
            )

            if token in column_lower:

                return column

    return None


def load_pairs(
    manifest_path
):

    df = pd.read_csv(
        manifest_path
    )

    image_col = find_column(
        df.columns,
        IMAGE_COLUMNS
    )

    label_col = find_column(
        df.columns,
        LABEL_COLUMNS
    )

    if (
        image_col is None
        or label_col is None
    ):

        raise ValueError(
            "Image/label columns could not "
            f"be detected in:\n{manifest_path}"
        )

    return pd.DataFrame(
        {
            "image_path":
                df[
                    image_col
                ].astype(str),

            "label_path":
                df[
                    label_col
                ].astype(str),
        }
    )


# ============================================================
# AUDIT
# ============================================================

audit_rows = []


for _, info in manifest_index.iterrows():

    pairs = load_pairs(
        info["manifest_path"]
    )

    print(
        f"\nChecking "
        f"{info['experiment_id']} "
        f"| {info['split']} "
        f"| {len(pairs)} pairs"
    )


    for _, row in pairs.iterrows():

        image_path = Path(
            row["image_path"]
        )

        label_path = Path(
            row["label_path"]
        )


        readable = False
        same_size = False

        foreground_percent = np.nan
        background_only = np.nan

        error = ""


        try:

            # --------------------------------------------
            # Existence check
            # --------------------------------------------

            if not image_path.exists():

                raise FileNotFoundError(
                    f"Missing image: "
                    f"{image_path}"
                )


            if not label_path.exists():

                raise FileNotFoundError(
                    f"Missing label: "
                    f"{label_path}"
                )


            # --------------------------------------------
            # Read
            # --------------------------------------------

            image = read_raster(
                image_path,
                is_mask=False
            )

            mask = read_raster(
                label_path,
                is_mask=True
            )


            if mask.ndim == 3:

                mask = mask[..., 0]


            readable = True


            # --------------------------------------------
            # Spatial size check
            # --------------------------------------------

            same_size = (
                image.shape[:2]
                ==
                mask.shape[:2]
            )


            # --------------------------------------------
            # Foreground building pixels
            # --------------------------------------------

            foreground = (
                mask > 0
            )


            foreground_percent = float(
                foreground.mean()
                * 100.0
            )


            background_only = bool(
                foreground.sum()
                == 0
            )


        except Exception as exc:

            error = str(exc)


        audit_rows.append(
            {
                "experiment_id":
                    info[
                        "experiment_id"
                    ],

                "split":
                    info["split"],

                "image_path":
                    str(image_path),

                "label_path":
                    str(label_path),

                "readable":
                    readable,

                "same_size":
                    same_size,

                "foreground_percent":
                    foreground_percent,

                "background_only":
                    background_only,

                "error":
                    error,
            }
        )


audit_df = pd.DataFrame(
    audit_rows
)


print(
    "\nDataset audit completed."
)

print(
    "Total pairs checked:",
    len(audit_df)
)

## 4. Dataset Quality Summary

This compact table provides the main quality indicators for each experiment and dataset split.

In [ ]:
summary_rows = []


for (
    experiment_id,
    split
), group in audit_df.groupby(
    [
        "experiment_id",
        "split"
    ]
):

    readable = group[
        group["readable"]
        == True
    ]


    summary_rows.append(
        {
            "Experiment":
                experiment_id,

            "Split":
                split,

            "Pairs":
                len(group),

            "Readable (%)":
                float(
                    group[
                        "readable"
                    ].mean()
                    * 100.0
                ),

            "Correct Size (%)":
                float(
                    group[
                        "same_size"
                    ].mean()
                    * 100.0
                ),

            "Background Tiles (%)":
                (
                    float(
                        readable[
                            "background_only"
                        ].mean()
                        * 100.0
                    )

                    if len(readable)

                    else np.nan
                ),

            "Mean Building Coverage (%)":
                (
                    float(
                        readable[
                            "foreground_percent"
                        ].mean()
                    )

                    if len(readable)

                    else np.nan
                ),
        }
    )


summary_df = pd.DataFrame(
    summary_rows
)


summary_df = (
    summary_df
    .sort_values(
        [
            "Experiment",
            "Split"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# SAVE ONLY THE IMPORTANT SUMMARY
# ============================================================

summary_path = (
    OUTPUT_DIR
    / "dataset_quality_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)


display(
    summary_df.style.format(
        {
            "Readable (%)":
                "{:.1f}",

            "Correct Size (%)":
                "{:.1f}",

            "Background Tiles (%)":
                "{:.1f}",

            "Mean Building Coverage (%)":
                "{:.2f}",
        }
    )
)


print(
    "\nSummary saved to:"
)

print(
    summary_path
)

## 5. Dataset Quality Overview

The following figure summarizes the proportion of background-only tiles.

Background tiles are useful as negative examples, but excessive background content can reduce training efficiency and create class imbalance.

In [ ]:
chart_df = summary_df.copy()


labels = (
    chart_df[
        "Experiment"
    ].astype(str)

    + "\n"

    + chart_df[
        "Split"
    ].astype(str)
)


values = chart_df[
    "Background Tiles (%)"
].fillna(0)


fig, ax = plt.subplots(
    figsize=(11, 5.5)
)


bars = ax.bar(
    labels,
    values
)


ax.set_ylabel(
    "Background-only tiles (%)"
)


ax.set_title(
    "Dataset Quality Overview — Background Tile Distribution"
)


ax.tick_params(
    axis="x",
    rotation=25
)


# Add value labels
for i, value in enumerate(
    values
):

    ax.text(
        i,
        value + 0.5,
        f"{value:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9
    )


fig.tight_layout()


quality_figure = (
    FIGURES_DIR
    / "dataset_quality_overview.png"
)


fig.savefig(
    quality_figure,
    dpi=220,
    bbox_inches="tight"
)


plt.show()


print(
    "Figure saved to:"
)

print(
    quality_figure
)

## 6. Representative Image–Mask Quality Check

A small number of representative samples is sufficient to visually verify that imagery and building labels are correctly aligned.

In [ ]:
eligible = audit_df[
    (
        audit_df["readable"]
        == True
    )
    &
    (
        audit_df["same_size"]
        == True
    )
].copy()


if eligible.empty:

    print(
        "No readable and aligned "
        "image-mask pairs available."
    )


else:

    samples = eligible.sample(
        n=min(
            SAMPLE_COUNT,
            len(eligible)
        ),
        random_state=RANDOM_SEED
    ).reset_index(
        drop=True
    )


    fig, axes = plt.subplots(
        len(samples),
        2,
        figsize=(
            11,
            4 * len(samples)
        ),
        squeeze=False
    )


    for i, row in samples.iterrows():

        image = read_raster(
            Path(
                row["image_path"]
            ),
            is_mask=False
        )


        mask = read_raster(
            Path(
                row["label_path"]
            ),
            is_mask=True
        )


        if mask.ndim == 3:

            mask = mask[..., 0]


        binary_mask = (
            mask > 0
        )


        # ================================================
        # Original image
        # ================================================

        axes[
            i,
            0
        ].imshow(
            image
        )


        axes[
            i,
            0
        ].set_title(
            f"{row['experiment_id']}\n"
            "Original Image"
        )


        axes[
            i,
            0
        ].axis(
            "off"
        )


        # ================================================
        # Image + mask
        # ================================================

        axes[
            i,
            1
        ].imshow(
            image
        )


        axes[
            i,
            1
        ].imshow(
            binary_mask,
            alpha=0.35,
            cmap="gray"
        )


        axes[
            i,
            1
        ].set_title(
            "Building Mask Overlay\n"
            f"Coverage: "
            f"{row['foreground_percent']:.2f}%"
        )


        axes[
            i,
            1
        ].axis(
            "off"
        )


    fig.tight_layout()


    sample_figure = (
        FIGURES_DIR
        / "representative_image_mask_check.png"
    )


    fig.savefig(
        sample_figure,
        dpi=200,
        bbox_inches="tight"
    )


    plt.show()


    print(
        "Figure saved to:"
    )

    print(
        sample_figure
    )

## 7. Final Dataset Quality Status

The final status summarizes whether the inspected datasets are technically suitable for model training and evaluation.

In [ ]:
# ============================================================
# QUALITY CHECK
# ============================================================

unreadable = int(
    (
        audit_df[
            "readable"
        ]
        == False
    ).sum()
)


size_mismatch = int(
    (
        (
            audit_df[
                "readable"
            ]
            == True
        )
        &
        (
            audit_df[
                "same_size"
            ]
            == False
        )
    ).sum()
)


# ============================================================
# FINAL STATUS
# ============================================================

if (
    unreadable == 0
    and size_mismatch == 0
):

    status = "PASS"

else:

    status = "REVIEW REQUIRED"


# ============================================================
# DISPLAY
# ============================================================

print(
    "=" * 62
)

print(
    "DATASET QUALITY AUDIT"
)

print(
    "=" * 62
)


print(
    f"Status: {status}"
)


print(
    f"Total pairs checked: "
    f"{len(audit_df)}"
)


print(
    f"Unreadable / missing pairs: "
    f"{unreadable}"
)


print(
    f"Image-label size mismatches: "
    f"{size_mismatch}"
)


readable_data = audit_df[
    audit_df[
        "readable"
    ]
    == True
]


if len(readable_data):

    overall_background = (
        readable_data[
            "background_only"
        ].mean()
        * 100.0
    )


    overall_coverage = (
        readable_data[
            "foreground_percent"
        ].mean()
    )


    print(
        f"Overall background-only tiles: "
        f"{overall_background:.1f}%"
    )


    print(
        f"Mean building coverage: "
        f"{overall_coverage:.2f}%"
    )


print(
    "=" * 62
)


if status == "PASS":

    print(
        "Conclusion:"
    )

    print(
        "The inspected datasets are technically "
        "consistent and suitable for model "
        "training and evaluation."
    )

else:

    print(
        "Conclusion:"
    )

    print(
        "Dataset-quality issues were detected. "
        "The affected samples should be reviewed "
        "before final experiments."
    )

## Output

Only the essential dataset-quality artifacts are retained:

```text
Buildings/
└── outputs/
    └── notebooks/
        └── 01_dataset_quality/
            ├── dataset_quality_summary.csv
            └── figures/
                ├── dataset_quality_overview.png
                └── representative_image_mask_check.png


That is the complete **Notebook 1**.

The main professional output should look conceptually like:

```text
DATASET QUALITY AUDIT
==============================================================
Status: PASS
Total pairs checked: ...
Unreadable / missing pairs: 0
Image-label size mismatches: 0
Overall background-only tiles: ...%
Mean building coverage: ...%
==============================================================

Conclusion:
The inspected datasets are technically consistent and suitable
for model training and evaluation.